[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/04_OCR_y_tagging.ipynb)


# 04 — PDF IMSS: OCR, comparación y **tagging**

1. Bajar el PDF y generar **`output/texto_pypdf.txt`**, **`texto_easyocr.txt`** y (opcional) tablas.
2. **Tagging en dos bloques separados**:
   - **A — Montos:** regex solo para cantidades con **`$`**, más el texto **entre paréntesis** (cantidad en letras). Resultado **por página** (tabla + resumen + JSON).
   - **B — Personas:** **NER** (`spaCy`), solo etiqueta **`PER`** (nombres propios; el modelo no separa género en la etiqueta).


In [ ]:
# ── Paso 1: instalar las librerías (solo la primera vez) ──────────────────────
# %pip install -q pypdf pymupdf easyocr pillow numpy matplotlib pdfplumber spacy

# ── Paso 2: descargar el modelo de español para spaCy ─────────────────────────
# El modelo NO viene incluido con spacy; hay que descargarlo por separado.
# Ejecuta UNA VEZ en terminal (o descomenta y corre aquí):
# %python -m spacy download es_core_news_sm

# ¿Por qué son dos pasos distintos?
#   pip install spacy   → instala la librería (el "motor")
#   spacy download ...  → descarga el modelo entrenado (los "datos")
# Son paquetes separados; sin el modelo, spacy.load() falla con un error de "not found".

Carpeta de trabajo: **`Modulo 4: NLP`**.

In [ ]:
# from pathlib import Path
# from subprocess import run

# URL_PDF = "https://reposipot.imss.gob.mx/contratos/CABCS/CTPC/DC/Contratos-Convenios/006400001N00225-001-00/006400001N00225-001-00_Censurado.pdf"

# CARPETA_SALIDA = Path("output")
# CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
# MAX_PAGINAS_OCR = 15
# ruta_pdf = CARPETA_SALIDA / "imss_documento.pdf"

# run(["curl", "-fL", "--progress-bar", "-A", "Mozilla/5.0", "-o", str(ruta_pdf), URL_PDF], check=True)

## 1) Texto con **pypdf** → `output/texto_pypdf.txt`

In [ ]:
from pypdf import PdfReader
from pathlib import Path
from subprocess import run

ruta_pdf = "data/006400001N00225-001-00_Censurado.pdf"
CARPETA_SALIDA = Path("output")

lector = PdfReader(ruta_pdf)
partes = []
for i, pagina in enumerate(lector.pages):
    partes.append(f"--- página {i + 1} ---\n")
    partes.append(pagina.extract_text() or "")
texto_pypdf = "\n".join(partes).strip()

salida_pypdf = CARPETA_SALIDA / "texto_pypdf.txt"
salida_pypdf.write_text(texto_pypdf, encoding="utf-8")
print(salida_pypdf.resolve(), "—", len(texto_pypdf), "caracteres")

## 2) Texto con **EasyOCR** (páginas como imagen) → `output/texto_easyocr.txt`

In [ ]:
# import easyocr
# import fitz
# import numpy as np
# from PIL import Image

# IDIOMAS = ["es", "en"]
# lector_ocr = easyocr.Reader(IDIOMAS, gpu=False)

# doc = fitz.open(ruta_pdf)
# n = min(doc.page_count, 10)  # solo primeras 10 páginas (OCR lento)
# bloques = []
# for i in range(n):
#     pix = doc.load_page(i).get_pixmap(matrix=fitz.Matrix(2, 2))
#     img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
#     det = lector_ocr.readtext(np.asarray(img), detail=1)
#     bloques.append(f"--- página {i + 1} ---\n" + "\n".join(t[1] for t in det))
# doc.close()

# texto_ocr = "\n\n".join(bloques).strip()
# salida_ocr = CARPETA_SALIDA / "texto_easyocr.txt"
# salida_ocr.write_text(texto_ocr, encoding="utf-8")
# print(salida_ocr.resolve(), "—", len(texto_ocr), "caracteres", f"({n} páginas OCR)")

## 3) (Opcional) **Tablas** con **pdfplumber** → `output/texto_tablas_pdfplumber.txt`

Intenta detectar tablas página por página y las vuelca como texto separado por `|` y líneas.

In [ ]:
import pdfplumber

lineas = []
with pdfplumber.open(ruta_pdf) as pdf:
    for i, pagina in enumerate(pdf.pages):
        lineas.append(f"--- página {i + 1} ---")
        tablas = pagina.extract_tables() or []
        if not tablas:
            lineas.append("(sin tablas detectadas en esta página)")
            continue
        for t_idx, tabla in enumerate(tablas):
            lineas.append(f"-- tabla {t_idx + 1} --")
            for fila in tabla:
                celdas = [str(c) if c is not None else "" for c in fila]
                lineas.append(" | ".join(celdas))

texto_tablas = "\n".join(lineas)
salida_tablas = CARPETA_SALIDA / "texto_tablas_pdfplumber.txt"
salida_tablas.write_text(texto_tablas, encoding="utf-8")
print(salida_tablas.resolve(), "—", len(texto_tablas), "caracteres")

## Resumen

Abre en el editor **`output/texto_pypdf.txt`** y **`output/texto_easyocr.txt`** y compara. Si `pypdf` está casi vacío y EasyOCR lleno, el PDF es sobre todo imagen escaneada.

---

## Cómo encajan **regex** y **NER**

| Qué quieres | Herramienta | Por qué |
|-------------|-------------|---------|
| Cantidades en pesos / dólares, formatos `1,234.56`, `$…`, `MN`… | **Regex** (`re`) | Los formatos se describen con reglas; no hace falta un modelo grande. |
| Quién firmó, qué empresa, instituciones, a veces fechas | **NER** | El modelo aprendió qué trozos de texto “parecen” persona u organización en español. |

**No compiten:** en un pipeline típico usas **ambos** sobre el mismo texto (el de `pypdf` o el de EasyOCR, el que mejor se lea).

**NER en la práctica (spaCy):** etiquetas útiles suelen ser `PER` (persona), `ORG` (organización), `LOC` (lugar); según el modelo puede haber más. No es magia: a veces confunde cargos con nombres; por eso en clase conviene **revisar** la lista.

**Instalación del modelo en español** (una vez en terminal o con `!python -m spacy download es_core_news_sm`):

```text
python -m spacy download es_core_news_sm
```

El sufijo `sm` = modelo pequeño (rápido). Para más calidad existe `md` / `lg` (más pesados).

## Tagging A — **Montos** (regex con **signo de pesos**)

Buscamos líneas del tipo `$8,800,000.00 (OCHO MILLONES … M.N.)`: el **número con `$`** y, si viene pegado, el **texto entre paréntesis**.

**Enfoque pedagógico:** primero probamos la **misma regla** en **una frase de ejemplo** (celdas siguientes). Cuando veas cómo encaja el patrón, la celda de abajo aplica **exactamente esa idea** a **`output/texto_pypdf.txt`** y guarda **`output/montos_por_pagina.json`**.


### Mini-ejemplo: **regex** en una frase

Misma regla que la celda de abajo: el **texto** y los **montos** que encuentra el patrón.


In [ ]:
import re

PAT_MONTO = re.compile(
    r"\$\s*[\d]{1,3}(?:,\d{3})*(?:\.\d{2})\s*(\([^)]+\))?",
    re.MULTILINE,
)

texto = "Pagar $1,250.50 hoy y luego $8,800,000.00 (OCHO MILLONES...) mañana."
texto, [m.group(0) for m in PAT_MONTO.finditer(texto)]


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

from IPython.display import display

CARPETA_SALIDA = Path("output")
RUTA_TEXTO = CARPETA_SALIDA / "texto_pypdf.txt"
texto = RUTA_TEXTO.read_text(encoding="utf-8")

# -----------------------------------------------------------------------------
# 1. Regla (regex): qué buscamos
#    - Empieza con $
#    - Número con comas de miles y dos decimales (ej. 8,800,000.00)
#    - Opcional: espacio y (… cantidad en letras …) como en el contrato IMSS
#    Si tu PDF usa otro formato, solo tocas PAT_MONTO aquí.
# -----------------------------------------------------------------------------
PAT_MONTO = re.compile(
    r"\$\s*[\d]{1,3}(?:,\d{3})*(?:\.\d{2})\s*(\([^)]+\))?",
    re.MULTILINE,
)

# -----------------------------------------------------------------------------
# 2. Cómo se parte el documento en "páginas"
#    pypdf ya metió marcas --- página N ---; esta función devuelve (n, texto).
# -----------------------------------------------------------------------------
def iter_paginas(contenido: str):
    marcas = list(re.finditer(r"---\s*página\s+(\d+)\s*---", contenido, flags=re.IGNORECASE))
    for i, m in enumerate(marcas):
        n = int(m.group(1))
        inicio = m.end()
        fin = marcas[i + 1].start() if i + 1 < len(marcas) else len(contenido)
        yield n, contenido[inicio:fin].strip()

# -----------------------------------------------------------------------------
# 3. Proceso: recorrer página por página y aplicar la regla
# -----------------------------------------------------------------------------
filas = []
por_pagina = []

for num_pag, bloque in iter_paginas(texto):
    hallazgos = []
    for m in PAT_MONTO.finditer(bloque):
        trozo = m.group(0).strip()
        en_palabras = (m.group(1) or "").strip()
        num_m = re.search(r"\$\s*[\d,]+\.\d{2}", trozo)
        monto_dolares = num_m.group(0).replace(" ", "") if num_m else trozo.split("(")[0].strip()
        filas.append(
            {"página": num_pag, "monto_$": monto_dolares, "entre_paréntesis": en_palabras}
        )
        hallazgos.append({"monto_$": monto_dolares, "entre_paréntesis": en_palabras})
    por_pagina.append({"página": num_pag, "montos": hallazgos})

# -----------------------------------------------------------------------------
# 4. Salida: archivo JSON en output/ + tabla + resumen en pantalla
# -----------------------------------------------------------------------------
salida_m = {"archivo_fuente": str(RUTA_TEXTO), "por_página": por_pagina}
(CARPETA_SALIDA / "montos_por_pagina.json").write_text(
    json.dumps(salida_m, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

df = pd.DataFrame(filas)
display(df)

print("\n--- Resumen por página ---\n")
for item in por_pagina:
    p = item["página"]
    lista = item["montos"]
    if not lista:
        continue
    print(f"Página {p}: se mencionan {len(lista)} monto(s) con $")
    for h in lista:
        pal = h["entre_paréntesis"]
        recorte = pal[:100] + ("…" if len(pal) > 100 else "")
        print("  ·", h["monto_$"], recorte)
    print()


## Tagging B — **Personas** (NER con spaCy)

Solo etiqueta **`PER`** (*person*): nombres propios que el modelo reconoce (hombre o mujer; el modelo no distingue por género en la etiqueta). Resultado **por página**.

**Antes** de procesar el PDF, las celdas siguientes muestran **qué es el NER** con una oración corta. **Después**, la última celda aplica el mismo modelo a **`output/texto_pypdf.txt`**.

La primera vez necesitas: `python -m spacy download es_core_news_sm`


### Mini-ejemplo: **NER** en una oración

Mismo modelo que la celda de abajo: la **oración** y las **entidades** `(texto, etiqueta)`.


In [ ]:
# %pip install spacy blis thinc --force-reinstall --no-cache-dir

### ¿Qué es `es_core_news_sm`?

El nombre del modelo no es aleatorio — cada parte indica exactamente qué contiene:

| Parte | Significado |
|-------|-------------|
| `es` | **Español** — idioma del modelo |
| `core` | Modelo **completo**: incluye tokenización, análisis gramatical (POS), dependencias sintácticas y NER |
| `news` | Entrenado con **textos de noticias** en español |
| `sm` | **Small** — versión pequeña y rápida |

---

#### ¿Qué tamaños existen?

spaCy ofrece tres tamaños para el modelo en español:

```
es_core_news_sm   # ~12 MB  → rápido, ideal para clase y prototipos
es_core_news_md   # ~43 MB  → incluye vectores de palabras (similitud semántica)
es_core_news_lg   # ~567 MB → más preciso en NER y análisis
```

> **Regla práctica:** empieza con `sm`. Si la precisión no es suficiente para tu caso de uso, sube a `md` o `lg`.

---

#### ¿Cómo instalo otro modelo?

```bash
python -m spacy download es_core_news_lg
```

Y en código solo cambias el nombre:

```python
nlp = spacy.load("es_core_news_lg")
```

Catálogo completo de modelos (todos los idiomas): https://spacy.io/models

In [ ]:
import spacy
from spacy import displacy

nlp_demo = spacy.load("es_core_news_sm")

oracion = (
    "La doctora Ana Pérez y el ingeniero Carlos Ruiz firmaron un convenio entre el IMSS "
    "y Microsoft México el pasado lunes en las oficinas de Guadalajara, "
    "representando al gobierno mexicano ante la ONU."
)

doc = nlp_demo(oracion)

# Visualización con colores (igual que en la imagen)
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
import spacy
from spacy import displacy
from IPython.display import display, HTML

CARPETA_SALIDA = Path("output")
texto = (CARPETA_SALIDA / "texto_pypdf.txt").read_text(encoding="utf-8")
nlp = spacy.load("es_core_news_sm")

def iter_paginas(contenido):
    marcas = list(re.finditer(r"---\s*página\s+(\d+)\s*---", contenido, flags=re.IGNORECASE))
    for i, m in enumerate(marcas):
        n = int(m.group(1))
        inicio = m.end()
        fin = marcas[i + 1].start() if i + 1 < len(marcas) else len(contenido)
        yield n, contenido[inicio:fin].strip()

filas = []
por_pagina = []

for num_pag, bloque in iter_paginas(texto):
    if not bloque.strip():
        continue
    doc = nlp(bloque[:50_000])  # displacy necesita docs más cortos
    entidades = [(ent.text.strip(), ent.label_) for ent in doc.ents if ent.text.strip()]
    vistos = set()
    nombres_unicos = []
    for texto_ent, label in entidades:
        if texto_ent not in vistos:
            vistos.add(texto_ent)
            filas.append({"página": num_pag, "entidad": texto_ent, "tipo": label})
            if label == "PER":
                nombres_unicos.append(texto_ent)
    por_pagina.append({"página": num_pag, "doc": doc, "personas": nombres_unicos})

# ── Tabla resumen ─────────────────────────────────────────────────────────────
display(HTML("<h3>Tabla de entidades por página</h3>"))
display(pd.DataFrame(filas))

# ── Visualización con colores por página ─────────────────────────────────────
display(HTML("<h3>Entidades resaltadas por página</h3>"))
for item in por_pagina:
    p = item["página"]
    doc = item["doc"]
    if not doc.ents:
        continue
    display(HTML(f"<h4 style='margin-top:1.5em;color:#444'>📄 Página {p}</h4>"))
    displacy.render(doc, style="ent", jupyter=True)
